# Ordinal Encoding
In this notebook I will explore the Frequency encoding method for every non-numerical feature. More specifically,

`"age"`: stays as it is; already numercial

`"workclass"`: makes no sense to make an Ordinal Encoding. However, one approach (altough not recomendable, it is just an idea) would be to use the order "<" wrt if a workclass on average makes more money (this must be done with outside data). Here my idea would be

>"Never-worked" < "Without-pay" < "Private" < "State-gov" < "Local-gov" < "Self-emp-not-inc" < "Federal-gov" < "Self-emp-inc"


`"fnlwgt"`: drop

`"education"`: Ordinal Encoding makes sense but is already done with `"education-num"`

`"education-num"`: stays as it is; already numercial

`"marital-status"`: makes no sense to make an Ordinal Encoding. However, one approach (altough not recomendable, it is just an idea) would be to use the order "<" wrt to the "seriousness" of the relationship (ie Married couple is more "serious" then just a couple). Here my idea would be

>"Priv-house-serv" < "Other-service" < "Handlers-cleaners" < "Farming-fishing" < "Machine-op-inspct" < "Transport-moving" < "Adm-clerical" < "Craft-repair" < "Sales" < "Tech-support" < "Protective-serv" < "Prof-specialty" < "Armed-Forces" < "Exec-managerial"

`"occupation"`:  makes no sense to make an Ordinal Encoding. However, one approach (altough not recomendable, it is just an idea) would be to use the order "<" wrt if a workclass on average makes more money (this must be done with outside data). Here my idea would be

>"Never-worked" < "Without-pay" < "Private" < "State-gov" < "Local-gov" < "Self-emp-not-inc" < "Federal-gov" < "Self-emp-inc"

`"relationship"`: makes no sense to make an Ordinal Encoding. However, one approach (altough not recomendable, it is just an idea) would be to use the order "<" wrt to the "seriousness" of the relationship (ie Married couple is more "serious" then just a couple). Here my idea would be

>"Not-in-family" < "Unmarried" < "Other-relative" < "Own-child" < "Husband" = "Wife"

`"race"`: makes no sense to make an Ordinal Encoding. I think even forcing a relation would be racist in this case.

`"sex"`: makes no sense to make an Ordinal Encoding. I think even forcing a relation would be sexist in this case.

`"capital-gain"`:stays as it is; already numercial

`"capital-loss"`: stays as it is; already numercial

`"hours-per-week"`: stays as it is; already numercial

`"native-country"`: makes no sense to make an Ordinal Encoding. However, one approach (altough not recomendable, it is just an idea) would be to use the order "<" wrt to the average income of each of these countries. This does not make sense generally because these people may not live in these countries. Here my idea would be

>"Haiti" < "Nicaragua" < "Laos" < "Honduras" < "India" < "Cambodia" < "Philippines" < "Vietnam" < "Guatemala" < "Iran" < "El-Salvador" < "Jamaica" < "Ecuador" < "Cuba" < "Columbia" < "Peru" < "Thailand" < "Dominican-Republic" < "Yugoslavia" < "Mexico" < "China" < "Trinadad&Tobago" < "Hungary" < "Poland" < "Greece" < "Portugal" < "Japan" < "Taiwan" < "South" < "Puerto-Rico" < "Outlying-US(Guam-USVI-etc)" < "Italy" < "France" < "England" < "Scotland" < "Germany" < "Canada" < "Hong" < "Holand-Netherlands" < "United-States" < "Ireland"

Here is the raw data loaded and apply
$$
y =
\begin{cases}
1 & \text{if income } > \$50K \\
0 & \text{otherwise}
\end{cases}
$$
to the data set. Also drop the columns which were stated before 

In [5]:
# Enable autoreloading of imported modules
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import sys
import os
import pandas as pd
from courselib.utils.splits import train_test_split

# Add the repo root (two levels up from this notebook) to sys.path
sys.path.insert(0, os.path.abspath("../../"))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
file_name = "adult.data"
column_names = column_names = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "Class"
]
df = pd.read_csv(file_name, names= column_names,skipinitialspace=True)

In [7]:
target_variable_map = {
    "<=50K": 0,
    ">50K": 1,
}

columns_to_be_dropped = [
    "fnlwgt",
    "education-num"
]

In [8]:
df= df.drop(columns=columns_to_be_dropped)
df["Class"] = df["Class"].map(target_variable_map)
df

,age,workclass,education,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,Class
0,39,State-gov,Bachelors,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,0
1,50,Self-emp-not-inc,Bachelors,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,0
2,38,Private,HS-grad,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,0
3,53,Private,11th,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,0
4,28,Private,Bachelors,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Private,Assoc-acdm,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,0
32557,40,Private,HS-grad,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,1
32558,58,Private,HS-grad,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,0
32559,22,Private,HS-grad,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,0


## Implementing a Ordinal encoding method (very simple)

In [ ]:
def Ordinal_Encoder(df,column_to_be_encoded, cols_to_order = [], categories_order =[[]], starting_numbers = None, step_sizes= None, drop_unknown_data_rows = [] ):
    """takes a pandas dataframe and columns which need to be encoded and creates transforms the columns accordingly to the other inputs. 
       outputs then a new edited copy of df.

    Args:
        df (pd.dataframe): just the dataset
        cols_to_order (list): a list of column names from df which need to be encoded
        categories_order (list of list): the length of the outer list is the same as cols_to_order, the inner list give the categories AND the order of these categories MUST be INCREASING. if a category is given as a list list itself, fe [cat1,cat2] they will be set equal
        starting_numbers (list of doubles, optional): list of numbers with the same length as cols_to_order, each number corresponds to the first entry of each inner list of categories_order. Defaults to starting each order with 1
        step_sizes (list of doubles, optional): list of numbers with the same length as cols_to_order, each number corresponds to the step size of each inner list of categories_order. Defaults to step size 1
        drop_unknown_data_rows (list, optional): Drops all rows which include the strings which are in this list. Defaults to empty list.
    """
    _df = df.copy()

    row_amount_before = len(_df)

    if type(column_to_be_encoded) == str:
        column_to_be_encoded = [column_to_be_encoded]

    if cols_to_order == []:
        cols_to_order = column_to_be_encoded

    if drop_unknown_data_rows !=[]:
        for column_name in cols_to_order:
            _df = _df[~_df[column_name].isin(drop_unknown_data_rows)]

    amount_of_rows_deleted = row_amount_before - len(_df)

    _df = _df.reset_index(drop=True)

    row_amount = len(_df)

    if starting_numbers is None:
        starting_numbers = [1 for _ in range(len(cols_to_order))]

    if step_sizes is None:
        step_sizes = [1 for _ in range(len(cols_to_order))]
    #handle errors here
    if len(cols_to_order) != len(categories_order):
        raise ValueError("len(cols_to_order) must be equal to len(categories_order)")

    if len(cols_to_order) != len(starting_numbers):
        raise ValueError("len(cols_to_order) must be equal to len(starting_numbers)")

    if len(cols_to_order)!= len(step_sizes):
        raise ValueError("len(cols_to_order) must be equal to len(step_sizes)")

    for i in range(len(cols_to_order)):
        column_name = cols_to_order[i]
        current_number = starting_numbers[i]
        replacement_dict = {}

        for category_or_list in categories_order[i]:
            if type(category_or_list) == list:
                for category in category_or_list:
                    replacement_dict[category] = current_number
            else:
                replacement_dict[category_or_list] = current_number

            current_number = current_number +step_sizes[i]

        old_column = _df[column_name].copy( )
        _df[column_name]= old_column.map(replacement_dict)

        if _df[column_name].isna().any():
            categories_not_in_order = list(old_column[_df[column_name].isna()].unique())
            raise ValueError(f"these categories from column '{column_name}' were not found in categories_order: {categories_not_in_order}")

    print(f"{amount_of_rows_deleted} out of {row_amount_before} were deleted, ie.{100*row_amount/row_amount_before}% still remain")

    return _df

Here are all inputs as we discussed in the beginning

In [10]:
column_to_be_encoded = [
    "education",
    "workclass",
    "relationship",
    "marital-status",
    "occupation",
    "native-country"
]

cols_to_order = [
    "education",
    "workclass",
    "relationship",
    "marital-status",
    "occupation",
    "native-country"
]

categories_order = [
    [
        "Preschool",
        "1st-4th",
        "5th-6th",
        "7th-8th",
        "9th",
        "10th",
        "11th",
        "12th",
        "HS-grad",
        "Some-college",
        "Assoc-voc",
        "Assoc-acdm",
        "Bachelors",
        "Masters",
        "Prof-school",
        "Doctorate"
    ],
    [
        "Never-worked",
        "Without-pay",
        "Private",
        "State-gov",
        "Local-gov",
        "Self-emp-not-inc",
        "Federal-gov",
        "Self-emp-inc"
    ],
    [
        "Not-in-family",
        "Unmarried",
        "Other-relative",
        "Own-child",
        ["Husband","Wife"]
    ],
    [
        "Never-married",
        "Separated",
        "Divorced",
        "Widowed",
        "Married-spouse-absent",
        ["Married-civ-spouse","Married-AF-spouse"]
    ],
    [
        "Priv-house-serv",
        "Other-service",
        "Handlers-cleaners",
        "Farming-fishing",
        "Machine-op-inspct",
        "Transport-moving",
        "Adm-clerical",
        "Craft-repair",
        "Sales",
        "Tech-support",
        "Protective-serv",
        "Prof-specialty",
        "Armed-Forces",
        "Exec-managerial"
    ],
    [
        "Haiti",
        "Nicaragua",
        "Laos",
        "Honduras",
        "India",
        "Cambodia",
        "Philippines",
        "Vietnam",
        "Guatemala",
        "Iran",
        "El-Salvador",
        "Jamaica",
        "Ecuador",
        "Cuba",
        "Columbia",
        "Peru",
        "Thailand",
        "Dominican-Republic",
        "Yugoslavia",
        "Mexico",
        "China",
        "Trinadad&Tobago",
        "Hungary",
        "Poland",
        "Greece",
        "Portugal",
        "Japan",
        "Taiwan",
        "South",
        "Puerto-Rico",
        "Outlying-US(Guam-USVI-etc)",
        "Italy",
        "France",
        "England",
        "Scotland",
        "Germany",
        "Canada",
        "Hong",
        "Holand-Netherlands",
        "United-States",
        "Ireland"
    ]
]

starting_numbers = [1,1,1,1,1,1]

step_sizes = [1,1,1,1,1,1]

In [11]:
from projectlib.encoding import drop_rows 

df_encoded = drop_rows(df,["?"])
df_encoded = Ordinal_Encoder(df_encoded,column_to_be_encoded,cols_to_order,categories_order,starting_numbers,step_sizes)

2399 out of 32561 were deleted, ie.92.63229016307852% still remain 
0 out of 30162 were deleted, ie.100.0% still remain


In [12]:
df_encoded

,age,workclass,education,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,Class
0,39,4,13,1,7,1,White,Male,2174,0,40,40,0
1,50,6,13,6,14,5,White,Male,0,0,13,40,0
2,38,3,9,3,3,1,White,Male,0,0,40,40,0
3,53,3,7,6,3,5,Black,Male,0,0,40,40,0
4,28,3,13,6,12,5,Black,Female,0,0,40,14,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
30157,27,3,12,6,10,5,White,Female,0,0,38,40,0
30158,40,3,9,6,5,5,White,Male,0,0,40,40,1
30159,58,3,9,4,7,2,White,Female,0,0,40,40,0
30160,22,3,9,1,7,4,White,Male,0,0,20,40,0


In [13]:
X, Y, X_train, Y_train, X_test, Y_test =  train_test_split(df_encoded, 
                                                           training_data_fraction=0.8, 
                                                           return_numpy=True)

print('Training data split as follows:')
print(f'  Training data samples: {len(X_train)}')
print(f'      Test data samples: {len(X_test)}')

Training data split as follows:
  Training data samples: 24130
      Test data samples: 6032
